In [1]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
import shap
import numpy as np

# Load and preprocess
df = pd.read_csv('Security_Vulnerabilities.csv')
severity_map = {'Low': 1, 'Moderate': 2, 'High': 3, 'Critical': 4}
df['Severity_Score'] = df['Severity'].map(severity_map)
df['Date'] = pd.to_datetime(df['Date'])
df['CVE_ID'] = df['Summary'].str.extract(r'(CVE-\d{4}-\d+)')

# Labels from scan_results.txt
with open('scan_results.txt', 'r') as f:
    lines = f.readlines()
priority_labels = [1 if 'Priority: Yes' in line else 0 for line in lines[::2]]
df_50 = df.iloc[:50].copy()
df_50['Priority'] = priority_labels

# Features
tfidf = TfidfVectorizer(max_features=100)
title_features = tfidf.fit_transform(df_50['Title']).toarray()
title_cols = [f'title_{i}' for i in range(title_features.shape[1])]
title_df = pd.DataFrame(title_features, columns=title_cols)
X = pd.concat([df_50[['Severity_Score']], title_df], axis=1)
y = df_50['Priority']

# Train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

# Accuracy
print(f"Model accuracy: {model.score(X_test_scaled, y_test):.2f}")

# SHAP
explainer = shap.LinearExplainer(model, X_train_scaled)
shap_values = explainer.shap_values(X_test_scaled)
shap.initjs()
shap.force_plot(explainer.expected_value, shap_values[0], X_test.iloc[0], feature_names=X.columns)

ValueError: Length of values (525) does not match length of index (50)